# Hyperparameter Optimization with Optuna

Optimizes LightGBM hyperparameters using **Optuna** (TPE sampler, 60 trials).

**Why this matters:** The default parameters used in `models.ipynb` were manually set.  
A committee can argue: *"Maybe macro features would help with better-tuned models."*  
This optimization applies to the **baseline feature set** — it closes that argument.

Search space:

| Parameter | Range |
|-----------|-------|
| `num_leaves` | 20 – 200 |
| `learning_rate` | 0.005 – 0.3 (log scale) |
| `min_child_samples` | 5 – 60 |
| `subsample` | 0.5 – 1.0 |
| `colsample_bytree` | 0.4 – 1.0 |
| `reg_alpha` | 1e-8 – 10 (log) |
| `reg_lambda` | 1e-8 – 10 (log) |
| `max_depth` | 4 – 12 |

Objective: minimize **validation RMSE** (original scale, not log scale).


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print(f'Optuna {optuna.__version__}')
except ImportError:
    raise ImportError("Install Optuna: pip install optuna")

import lightgbm as lgb
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 130
os.makedirs('plots/tuning', exist_ok=True)
os.makedirs('saved_models', exist_ok=True)

BASE = '/Users/kamilaya/Desktop/thesis'

def rmse_orig(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(
        np.expm1(np.asarray(y_true_log)),
        np.expm1(np.clip(np.asarray(y_pred_log), 0, None))
    ))

def nz_mape(a, b):
    a, b = np.asarray(a), np.asarray(b)
    mask = a > 0
    return np.mean(np.abs((a[mask]-b[mask])/a[mask]))*100 if mask.sum() else np.nan


Optuna 4.8.0


In [4]:
train = pd.read_csv(f'{BASE}/tshirts_train.csv', parse_dates=['week_start'])
val   = pd.read_csv(f'{BASE}/tshirts_val.csv',   parse_dates=['week_start'])
test  = pd.read_csv(f'{BASE}/tshirts_test.csv',  parse_dates=['week_start'])
TARGET = 'log_sales_volume'

OHE_COLS = [c for c in train.columns if any(c.startswith(p) for p in [
    'index_group_name_', 'colour_group_name_',
    'graphical_appearance_name_', 'perceived_colour_value_name_'])]

BASE_FEATURES = [
    'avg_weekly_price', 'real_price', 'price_vs_median',
    'week_of_year', 'month', 'quarter', 'is_spring_summer', 'is_sale_season', 'covid',
    'product_age_weeks', 'sales_lag1', 'sales_lag2', 'sales_lag4',
    'sales_rolling4_mean', 'sales_rolling4_std',
] + OHE_COLS
features = [f for f in BASE_FEATURES if f in train.columns]

X_train = train[features]; y_train = train[TARGET]
X_val   = val[features];   y_val   = val[TARGET]
X_test  = test[features];  y_test  = test[TARGET]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Features: {len(features)}')

# Default model baseline (from models.ipynb) for comparison
DEFAULT_VAL_RMSE  = 5.2051
DEFAULT_TEST_RMSE = 6.2436
print(f'\nDefault LightGBM (models.ipynb): Val RMSE={DEFAULT_VAL_RMSE}  Test RMSE={DEFAULT_TEST_RMSE}')


Train: 259,809  Val: 275,555  Test: 299,174
Features: 97

Default LightGBM (models.ipynb): Val RMSE=5.2051  Test RMSE=6.2436


In [5]:
def objective(trial):
    params = {
        'n_estimators':      1000,          # controlled by early stopping
        'learning_rate':     trial.suggest_float('learning_rate',     0.005, 0.3,  log=True),
        'num_leaves':        trial.suggest_int(  'num_leaves',        20,    200),
        'max_depth':         trial.suggest_int(  'max_depth',         4,     12),
        'min_child_samples': trial.suggest_int(  'min_child_samples', 5,     60),
        'subsample':         trial.suggest_float('subsample',         0.5,   1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree',  0.4,   1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha',         1e-8,  10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda',        1e-8,  10.0, log=True),
        'random_state': 42, 'verbose': -1, 'n_jobs': -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    return rmse_orig(y_val, model.predict(X_val))


print('Starting Optuna search (60 trials)...')
print('Expected runtime: ~15-25 min on CPU')
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner()
)
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f'\nBest val RMSE: {study.best_value:.4f}  (default: {DEFAULT_VAL_RMSE})')
print(f'Improvement:   {(DEFAULT_VAL_RMSE - study.best_value) / DEFAULT_VAL_RMSE * 100:+.2f}%')
print(f'\nBest parameters:')
for k, v in study.best_params.items():
    print(f'  {k:<22} = {v}')


Starting Optuna search (60 trials)...
Expected runtime: ~15-25 min on CPU


Best trial: 49. Best value: 5.17512: 100%|██████████| 60/60 [14:18<00:00, 14.31s/it]


Best val RMSE: 5.1751  (default: 5.2051)
Improvement:   +0.58%

Best parameters:
  learning_rate          = 0.02040570058775951
  num_leaves             = 153
  max_depth              = 10
  min_child_samples      = 52
  subsample              = 0.8520324598005982
  colsample_bytree       = 0.6546609561123533
  reg_alpha              = 0.00023723124429476597
  reg_lambda             = 0.06549409673164905


In [6]:
# Retrain on train+val with best params and evaluate on test
best_params = study.best_params.copy()
best_params.update({'n_estimators': 2000, 'random_state': 42, 'verbose': -1, 'n_jobs': -1})

tuned_model = lgb.LGBMRegressor(**best_params)
tuned_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)

y_val_raw  = np.expm1(y_val.values)
y_test_raw = np.expm1(y_test.values)

tuned_val_pred  = np.expm1(np.clip(tuned_model.predict(X_val), 0, None))
tuned_test_pred = np.expm1(np.clip(tuned_model.predict(X_test), 0, None))

from sklearn.metrics import mean_absolute_error as mae_fn
print('='*60)
print('TUNED vs DEFAULT LightGBM — FINAL COMPARISON')
print('='*60)
print(f'{"":30s} {"Val RMSE":>10} {"Test RMSE":>10} {"Test MAE":>10}')
print(f'{"Default (models.ipynb)":30s} {DEFAULT_VAL_RMSE:>10.4f} {DEFAULT_TEST_RMSE:>10.4f} {"1.0702":>10}')

tuned_v = np.sqrt(mean_squared_error(y_val_raw, tuned_val_pred))
tuned_t = np.sqrt(mean_squared_error(y_test_raw, tuned_test_pred))
print(f'{"Tuned (Optuna 60 trials)":30s} {tuned_v:>10.4f} {tuned_t:>10.4f} {mae_fn(y_test_raw, tuned_test_pred):>10.4f}')
print(f'{"Improvement":30s} {(DEFAULT_VAL_RMSE-tuned_v)/DEFAULT_VAL_RMSE*100:>+9.2f}% {(DEFAULT_TEST_RMSE-tuned_t)/DEFAULT_TEST_RMSE*100:>+9.2f}%')

joblib.dump(tuned_model, f'{BASE}/saved_models/LightGBM_tuned.pkl')
print('\nTuned model saved: saved_models/LightGBM_tuned.pkl')


TUNED vs DEFAULT LightGBM — FINAL COMPARISON
                                 Val RMSE  Test RMSE   Test MAE
Default (models.ipynb)             5.2051     6.2436     1.0702
Tuned (Optuna 60 trials)           5.1751     6.2440     1.0812
Improvement                        +0.58%     -0.01%

Tuned model saved: saved_models/LightGBM_tuned.pkl


In [7]:
# ── Plot 1: Optimization history ──────────────────────────────────────────────
trials_df = study.trials_dataframe()
trials_df['best_so_far'] = trials_df['value'].cummin()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(trials_df.index, trials_df['value'], alpha=0.45, s=25, color='steelblue', label='Trial RMSE')
axes[0].plot(trials_df.index, trials_df['best_so_far'], color='tomato', linewidth=2.5, label='Best so far')
axes[0].axhline(DEFAULT_VAL_RMSE, color='orange', linestyle='--', linewidth=1.5, label=f'Default ({DEFAULT_VAL_RMSE})')
axes[0].set_xlabel('Trial'); axes[0].set_ylabel('Val RMSE')
axes[0].set_title('Optuna Optimization History\n(LightGBM, 60 trials)', fontweight='bold')
axes[0].legend()

# ── Plot 2: Hyperparameter importance ─────────────────────────────────────────
importance = optuna.importance.get_param_importances(study)
params_sorted = list(importance.keys())
vals_sorted   = [importance[k] for k in params_sorted]

axes[1].barh(params_sorted[::-1], vals_sorted[::-1], color='mediumseagreen', edgecolor='white', alpha=0.88)
axes[1].set_xlabel('Relative importance')
axes[1].set_title('Hyperparameter Importance\n(Optuna fANOVA estimator)', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/tuning/H1_optuna_history.png')
plt.close()

# ── Plot 3: Val RMSE distribution across trials ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(trials_df['value'].dropna(), bins=20, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(study.best_value,     color='tomato',  linewidth=2, label=f'Best tuned: {study.best_value:.4f}')
ax.axvline(DEFAULT_VAL_RMSE,     color='orange',  linewidth=2, linestyle='--', label=f'Default: {DEFAULT_VAL_RMSE}')
ax.set_xlabel('Val RMSE'); ax.set_ylabel('Count')
ax.set_title('Distribution of Trial RMSE Values', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('plots/tuning/H2_rmse_distribution.png')
plt.close()

print('Plots saved to plots/tuning/')


Plots saved to plots/tuning/


In [8]:
print('='*60)
print('HYPERPARAMETER TUNING SUMMARY')
print('='*60)
print(f'Trials run:        60')
print(f'Best val RMSE:     {study.best_value:.4f}')
print(f'Default val RMSE:  {DEFAULT_VAL_RMSE}')
pct = (DEFAULT_VAL_RMSE - study.best_value) / DEFAULT_VAL_RMSE * 100
print(f'Val improvement:   {pct:+.2f}%')
print()
print('Interpretation:')
print('  Hyperparameter tuning validates that the default parameter choice')
print('  in models.ipynb was already near-optimal. Any remaining gap between')
print('  ML and non-ML baselines cannot be attributed to poor hyperparameters.')
print()
print('Best hyperparameters:')
for k, v in study.best_params.items():
    default_val = {
        "learning_rate": 0.05, "num_leaves": 63, "max_depth": -1,
        "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_alpha": 0.0, "reg_lambda": 0.0
    }.get(k, "N/A")
    print(f'  {k:<22} = {str(v):<15}  (default: {default_val})')


HYPERPARAMETER TUNING SUMMARY
Trials run:        60
Best val RMSE:     5.1751
Default val RMSE:  5.2051
Val improvement:   +0.58%

Interpretation:
  Hyperparameter tuning validates that the default parameter choice
  in models.ipynb was already near-optimal. Any remaining gap between
  ML and non-ML baselines cannot be attributed to poor hyperparameters.

Best hyperparameters:
  learning_rate          = 0.02040570058775951  (default: 0.05)
  num_leaves             = 153              (default: 63)
  max_depth              = 10               (default: -1)
  min_child_samples      = 52               (default: 20)
  subsample              = 0.8520324598005982  (default: 0.8)
  colsample_bytree       = 0.6546609561123533  (default: 0.8)
  reg_alpha              = 0.00023723124429476597  (default: 0.0)
  reg_lambda             = 0.06549409673164905  (default: 0.0)
